# Week 4 Capstone — RAG-as-a-Tool Chat Assistant

Hybrid RAG (FAISS + BM25) over two PDFs, wrapped as a single tool that a raw OpenAI `gpt-4o-mini` tool-calling loop invokes only when a question is actually about the documents.

## 1. RAG Tool

Loads two PDFs, chunks them, and builds a hybrid retriever (FAISS for semantic search + BM25 for keyword search, combined via `EnsembleRetriever`). Retrieval and generation are wrapped into one function, `search_documents(query)`, which returns an answer with page-level `[Source : ...]` citations built from the PDFs' own metadata. Uses LangChain + Ollama only.

In [1]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters langchain-classic faiss-cpu pypdf rank_bm25 openai python-dotenv -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_ollama import OllamaEmbeddings, ChatOllama

C:\Users\shiva\AppData\Local\Temp\ipykernel_14464\1652787523.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PDF_FILES = ["A._P._J._Abdul_Kalam.pdf", "Gopalswamy Doraiswamy Naidu - Wikipedia.pdf"]

pages = []
for pdf_file in PDF_FILES:
    pages.extend(PyPDFLoader(pdf_file).load())

print(f"Total pages loaded: {len(pages)}")

Total pages loaded: 34


### Clean citation metadata

Each PDF is a browser print-to-PDF of a Wikipedia article, so `pypdf` already picks up a clean `title` (e.g. `"A. P. J. Abdul Kalam - Wikipedia"`) and a 1-indexed `page_label` from the PDF itself. We just tidy the title and store both under simple names so they ride along through chunking.

In [4]:
for doc in pages:
    title = doc.metadata.get("title", "")
    fallback_title = doc.metadata["source"].rsplit(".", 1)[0].replace("_", " ")
    doc.metadata["doc_title"] = title.replace(" - Wikipedia", "").strip() if title else fallback_title
    doc.metadata["page_number"] = doc.metadata.get("page_label", doc.metadata["page"] + 1)

print(pages[0].metadata["doc_title"], "| page", pages[0].metadata["page_number"])

A. P. J. Abdul Kalam | page 1


In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)}")

Total chunks: 149


### Hybrid retriever + generation LLM

FAISS (semantic) and BM25 (keyword) each retrieve independently, then `EnsembleRetriever` merges their rankings. `rag_llm` and `RAG_PROMPT` are the (Ollama) generation side of the tool.

In [6]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = FAISS.from_documents(chunks, embeddings)

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)

rag_llm = ChatOllama(model="llama3.2:3b", temperature=0)

RAG_PROMPT = """Answer the question using only the numbered context chunks below.

{context}

Question: {query}

Respond in exactly this format:
Answer: <your answer>
Used: <comma-separated numbers of only the chunks you actually used, e.g. 1,3>"""

print(f"Vector store: {vector_store.index.ntotal} vectors | Hybrid retriever ready")

Vector store: 149 vectors | Hybrid retriever ready


### `search_documents` — retrieval + generation + citations in one function

Context chunks are numbered, and the LLM reports back which chunk numbers it actually used to answer — citations are then built only from those chunks' `doc_title` / `page_number` metadata, so a chunk that was merely *retrieved* (hybrid search noise) but not used in the answer never shows up as a source.

In [7]:
def search_documents(query: str) -> str:
    docs = hybrid_retriever.invoke(query)
    context = "\n\n".join(f"[{i}] {doc.page_content}" for i, doc in enumerate(docs, start=1))

    raw = rag_llm.invoke(RAG_PROMPT.format(context=context, query=query)).content
    answer_part, _, used_part = raw.partition("Used:")
    answer = answer_part.replace("Answer:", "", 1).strip()

    used_numbers = [int(n) for n in used_part.replace(",", " ").split() if n.isdigit()]
    used_docs = [docs[n - 1] for n in used_numbers if 1 <= n <= len(docs)]
    if not used_docs:
        used_docs = docs[:1]  # fallback if the model didn't report any usable numbers

    seen = set()
    citations = []
    for doc in used_docs:
        key = (doc.metadata["doc_title"], doc.metadata["page_number"])
        if key not in seen:
            seen.add(key)
            citations.append(f"[Source : {key[0]}, Page {key[1]}]")

    return answer + "\n\n" + "\n".join(citations)

### Standalone test — before wiring the tool into the assistant

In [8]:
test_output = search_documents("What missile programs did Abdul Kalam lead at DRDO?")
print(test_output)

Integrated Guided Missile Development Programme (IGMDP)

[Source : A. P. J. Abdul Kalam, Page 4]


## 2. Tool-Calling Assistant

A raw OpenAI SDK chat loop (`gpt-4o-mini`, no agent framework) that keeps real conversation history. It calls the `search_documents` tool only when a question is actually about the two documents, and answers directly otherwise. When the tool is called, the model's own generated arguments are ignored — the tool always receives the user's original question, unrewritten.

In [9]:
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

MODEL = "gpt-4o-mini"
client = OpenAI()

In [10]:
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "search_documents",
        "description": "Search the two loaded documents (A. P. J. Abdul Kalam and G. D. Naidu biographies) to answer questions about them.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The question to search for"
                }
            },
            "required": ["query"]
        }
    }
}

SYSTEM_PROMPT = """You are a helpful assistant.
Call the search_documents tool when the user asks something about A. P. J. Abdul Kalam or G. D. Naidu.
For general questions or small talk, answer directly without using the tool.
If a tool result contains a line starting with "[Source :", keep that line exactly as given at the end of your reply."""

### Chat loop — simulated conversation

2 turns test conversation memory, 2 ask about PDF 1 (Kalam), 2 ask about PDF 2 (Naidu), ending in `"quit"`.

In [11]:
user_inputs = [
    "Hi, I'm Shiva and I'm learning GenAI development.",       # memory
    "What's my name and what am I learning?",                  # memory recall
    "What missile programs did Abdul Kalam lead at DRDO?",     # pdf1
    "When did Kalam become President of India and for how long?",  # pdf1
    "What did G. D. Naidu invent in India?",                   # pdf2
    "Where was G. D. Naidu born and when did he die?",         # pdf2
    "quit",
]

available_functions = {"search_documents": search_documents}
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

for user_input in user_inputs:
    if user_input == "quit":
        break

    print(f"\n{'=' * 60}\nUSER: {user_input}\n{'=' * 60}")
    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(model=MODEL, messages=messages, tools=[SEARCH_TOOL])
    choice = response.choices[0]

    if choice.finish_reason == "tool_calls":
        tool_call = choice.message.tool_calls[0]
        print(f"Tool called: {tool_call.function.name}")

        tool_result = available_functions[tool_call.function.name](user_input)
        print(f"Tool output:\n{tool_result}")

        messages.append(choice.message)
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": tool_result})

        final_response = client.chat.completions.create(model=MODEL, messages=messages)
        final_message = final_response.choices[0].message
    else:
        print("Tool called: none")
        final_message = choice.message

    messages.append(final_message)
    print(f"Final response:\n{final_message.content}")


USER: Hi, I'm Shiva and I'm learning GenAI development.


Tool called: none
Final response:
Hi Shiva! That sounds like an exciting field to be diving into. GenAI development offers a lot of opportunities and innovations. What specific areas of GenAI are you interested in, or is there something particular you're working on?

USER: What's my name and what am I learning?


Tool called: none
Final response:
Your name is Shiva and you're learning GenAI development.

USER: What missile programs did Abdul Kalam lead at DRDO?


Tool called: search_documents


Tool output:
Integrated Guided Missile Development Programme (IGMDP)

[Source : A. P. J. Abdul Kalam, Page 4]


...

Tool output:
The first electric motor in India.

[Source : Gopalswamy Doraiswamy Naidu, Page 1]


Final response:
G. D. Naidu is credited with inventing the first electric motor in India. 

[Source : Gopalswamy Doraiswamy Naidu, Page 1]

USER: Where was G. D. Naidu born and when did he die?


Tool called: search_documents


Tool output:
Kalangal, Coimbatore District, Madras Presidency, British India (23 March 1893), Coimbatore, Tamil Nadu, India (4 January 1974)

[Source : Gopalswamy Doraiswamy Naidu, Page 1]


Final response:
G. D. Naidu was born in Kalangal, Coimbatore District, Madras Presidency, British India, on 23 March 1893. He passed away in Coimbatore, Tamil Nadu, India, on 4 January 1974.

[Source : Gopalswamy Doraiswamy Naidu, Page 1]
